# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
# Print basic info
print(f"Dataset Loaded: {getattr(dataset.metadata, 'name', None)}\nDescription: {getattr(dataset.metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant datasets organize tabular data into **record sets**. Each record set is uniquely identified by an `@id`. Inside a record set, you'll find fields (columns) also identified by their own `@id`.

Let's enumerate all record sets and preview the fields and their IDs. This information will help us reference the correct IDs for further data extraction and analysis.

In [ ]:
# List available record sets by their @id
record_set_objs = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_set_objs)}")
all_record_set_ids = []
record_set_fields = {}
for rs in record_set_objs:
    rs_id = getattr(rs, '@id', None)
    all_record_set_ids.append(rs_id)
    print(f"\nRecord Set @id: {rs_id}")
    print(f"  Name       : {getattr(rs, 'name', '')}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    # Fields (columns)
    fields = getattr(rs, 'fields', [])
    field_ids = []
    for field in fields:
        field_id = getattr(field, '@id', None)
        field_name = getattr(field, 'name', None)
        print(f"    - Field: {field_name}, @id: {field_id}, type: {getattr(field, 'data_type', None)}")
        field_ids.append(field_id)
    record_set_fields[rs_id] = field_ids

if not all_record_set_ids:
    print("No record sets found in the dataset metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

The following code will extract all rows from each record set (referenced by their `@id`s) and convert them to pandas DataFrames for analysis.

In [ ]:
# Extract data from each record set into DataFrames
dataframes = {}
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id} -> shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print()
# For this example, automatically choose the first record set for our analyses
main_record_set_id = all_record_set_ids[0] if all_record_set_ids else None
if main_record_set_id:
    print(f"Using record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print("No record set available for further extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll:
- Select a numeric field via its `@id` (choose from previously listed fields)
- Filter the DataFrame based on a threshold value in this field
- Normalize the field
- Optionally group by a categorical field (when available)

The next cell infers the numeric and categorical columns present in your main record set.

In [ ]:
# Identify a numeric field for EDA
import numpy as np

df = dataframes.get(main_record_set_id)
if df is not None and not df.empty:
    # Try to auto-detect numeric columns
    possible_numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not possible_numeric_fields:
        # Try parsing columns to float if possible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                pass
        possible_numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"Chosen numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()  # Example threshold: mean
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Rows with {numeric_field} > {threshold:.2f}: {len(filtered_df)}")
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try a grouping by a likely categorical field
        possible_categoricals = [c for c in df.columns if c != numeric_field and df[c].nunique() < df.shape[0]//2]
        if possible_categoricals:
            group_field = possible_categoricals[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            group_field = None
    else:
        numeric_field = None
        group_field = None
        print("No numeric fields found for EDA.")
else:
    numeric_field, group_field = None, None
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field and its normalized values. If a group field is available, we plot the average value per group.

In [ ]:
if df is not None and numeric_field:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if f"{numeric_field}_normalized" in filtered_df:
        plt.figure(figsize=(10,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True)
        plt.title(f"Distribution of Normalized {numeric_field} (filtered)")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,4))
        group_means = filtered_df.groupby(group_field)[numeric_field].mean()
        group_means = group_means.sort_values(ascending=False)
        sns.barplot(x=group_means.index.astype(str), y=group_means.values)
        plt.xticks(rotation=45)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric or group field found for visualization.")

## 6. Conclusion
In this notebook, we have:
- Loaded and parsed dataset metadata using the `mlcroissant` library and its Croissant schema
- Reviewed the data structure, including record sets and field IDs
- Extracted tabular data for analysis
- Performed simple filtering, normalization, grouping, and visualization operations using fields referenced via their `@id`

For more advanced tasks, you can utilize the precise record set and field `@id`s, and extend this notebook with tailored data processing and ML pipelines.